# Real World Validation

Goal:
Evaluate whether the baseline model generalizes.

Dataset:
Custom recordings

Questions:

- Does model survive outside ESC50?
- Which classes fail?

In [84]:
from pathlib import Path

import numpy as np

import librosa

import tensorflow_hub as hub

import joblib

import pandas as pd

In [85]:
clf = joblib.load(
    "../models/baseline.pkl"
)

yamnet = hub.load(
    "https://tfhub.dev/google/yamnet/1"
)

## Predict Sound

In [86]:
def predict_audio(path):

    waveform, sr = librosa.load(
        path,
        sr=16000
    )

    waveform = waveform.astype(
        np.float32
    )

    scores,\
    embeddings,\
    spec = yamnet(
        waveform
    )

    emb = (
        embeddings
        .numpy()
        .mean(
            axis=0
        )
    )

    pred = clf.predict(
        [emb]
    )[0]

    conf = (
        clf
        .predict_proba(
            [emb]
        )
        .max()
    )

    return pred, conf

In [87]:
sample = (
    "../data/"
    "real_world/"
    "audio/"
    "crying_baby/"
    "baby_2.wav"
)

In [88]:
predict_audio(
    sample
)

(np.str_('siren'), np.float32(0.6203565))

## Evaluate Folder

In [89]:
ROOT = Path(
    "../data/real_world/audio"
)

results = []

In [90]:
for label in ROOT.iterdir():

    for file in label.glob(
        "*.wav"
    ):

        pred,\
        conf = predict_audio(
            file
        )

        results.append({

            "true": label.name,

            "pred": pred,

            "confidence": conf

        })

In [91]:
df = pd.DataFrame(
    results
)

df

,true,pred,confidence
0,clock_alarm,clock_alarm,0.989734
1,clock_alarm,clock_alarm,0.999589
2,clock_alarm,clock_alarm,0.704437
3,clock_alarm,clock_alarm,0.995379
4,clock_alarm,clock_alarm,0.999868
5,clock_alarm,clock_alarm,0.999259
6,clock_alarm,clock_alarm,0.998181
7,clock_alarm,clock_alarm,0.999979
8,clock_alarm,clock_alarm,0.999987
9,clock_alarm,clock_alarm,0.998822


In [92]:
(
    df["true"]
    ==
    df["pred"]
).mean()

np.float64(0.9487179487179487)

In [93]:
df[
    df["true"]
    !=
    df["pred"]
]

,true,pred,confidence
12,crying_baby,siren,0.620357
31,siren,door_wood_knock,0.734543


# Findings

## Real World Accuracy

0.9487 (94.87%)

Result:
Performance remained strong outside the training dataset.

---

## Observations

Most sounds generalized correctly.

A small number of recordings failed.

Expected degradation occurred compared to cross validation.

Cross Validation:
99.38%

Real World:
94.87%

Gap:
≈ 4.5%

---

## Interpretation

The model appears to learn sound characteristics rather than memorizing ESC50.

Performance is promising for a first baseline.

---

## Limitations

Validation size remains small.

Real environments are still limited.

Long-term robustness unknown.

---

## Decision

Accept baseline.

Proceed toward deployment preparation.